**Part 3 · LangGraph** · step 17 of 25  
[← 16 First graph](16_first_graph.ipynb) · [🏠 Roadmap](../README.md) · [18 Tool-calling agent graph →](18_tool_calling_agent_graph.ipynb)

# 17 · State & Reducers — what does `Annotated[list, add_messages]` mean?

When a node returns `{"key": value}`, LangGraph must decide how to combine it with the old value.
That rule is the **reducer**.

- **No reducer** → the new value **replaces** the old one.
- **With a reducer** → `new_state = reducer(old_value, returned_value)`.

You attach a reducer with `typing.Annotated[type, reducer_function]`.
`Annotated` just attaches extra metadata to a type hint — LangGraph reads that metadata.

| Annotation | Behaviour |
|---|---|
| `count: int` | overwrite |
| `items: Annotated[list, operator.add]` | concatenate lists (`old + new`) |
| `messages: Annotated[list[AnyMessage], add_messages]` | append messages, **update in place if the id already exists**, accepts dicts/tuples |

*Runs offline — no API key needed.*

## 1. Overwrite vs. `operator.add`

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    """``last`` is overwritten, ``history`` accumulates."""

    last: str
    history: Annotated[list[str], operator.add]


def step_a(state: State) -> dict:
    """Return one item for each key."""
    return {"last": "A", "history": ["A"]}


def step_b(state: State) -> dict:
    """Same keys again — watch how each one is merged."""
    return {"last": "B", "history": ["B"]}


builder = StateGraph(State)
builder.add_node("step_a", step_a)
builder.add_node("step_b", step_b)

builder.add_edge(START, "step_a")
builder.add_edge("step_a", "step_b")
builder.add_edge("step_b", END)

builder.compile().invoke({"last": "", "history": ["start"]})
# -> last == "B"  (replaced),  history == ["start", "A", "B"]  (concatenated)

{'last': 'B', 'history': ['start', 'A', 'B']}

## 2. `add_messages` — the reducer for chat history

In [10]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages

old = [HumanMessage("hi", id="1")]

# Appends new messages
print(add_messages(old, [AIMessage("hello!", id="2")]))


# print(old)
print("-------------------")
# Same id -> REPLACES that message instead of appending
print(add_messages(old, [HumanMessage("hi (edited)", id="1")]))

print(old)
# Dicts are converted to message objects automatically
print(add_messages(old, [{"role": "ai", "content": "from a dict"}]))

[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='1'), AIMessage(content='hello!', additional_kwargs={}, response_metadata={}, id='2', tool_calls=[], invalid_tool_calls=[])]
-------------------
[HumanMessage(content='hi (edited)', additional_kwargs={}, response_metadata={}, id='1')]
[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='1')]
[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='1'), AIMessage(content='from a dict', additional_kwargs={}, response_metadata={}, id='f31d9794-6c71-4684-a1d5-8985c3ff4100', tool_calls=[], invalid_tool_calls=[])]


## ⚠️ Common mistake
With `add_messages`, a node should return **only the new messages**:
```python
return {"messages": [llm.invoke(state["messages"])]}                       # ✅
return {"messages": state["messages"] + [llm.invoke(state["messages"])]}   # ❌ resends everything
```
The second version only "works" because existing ids are de-duplicated — with `operator.add` it would duplicate the whole history.

### `MessagesState` shortcut
`from langgraph.graph import MessagesState` is a ready-made state with
`messages: Annotated[list[AnyMessage], add_messages]`. Subclass it to add your own keys.

## 3. Where are reducers used in real production systems?

A reducer answers one simple question: **"When new data comes in, what do I do with the old data?"**

| Real-life use case | Reducer | What it does |
|---|---|---|
| **Chatbot / customer support bot** remembers the whole conversation | `add_messages` | adds each new message to the chat history |
| **Edit or delete messages** (user fixes a typo, admin removes bad message) | `add_messages` (same id / `RemoveMessage`) | replaces or deletes one message |
| **Keep chat short to save LLM cost** | `add_messages` + `RemoveMessage` | deletes old messages so fewer tokens are sent |
| **Parallel workers** (search web + docs + database at the same time) | `operator.add` | collects results from all workers into one list |
| **Logs / audit trail** (which steps ran, which errors happened) | `operator.add` | keeps adding entries, never loses old ones |
| **Cost tracking** (total tokens used in one run) | custom: `old + new` | adds up numbers |
| **Form filling** (one node finds name, another finds address) | custom: merge dicts | combines fields from many nodes |

> **Simple rule:** if a key should be **replaced** each time (like `current_step`), don't use a reducer.
> If a key should **collect or combine** data (like `messages`, `results`, `total_tokens`), use a reducer.


### 3a. Parallel workers: collect results from many nodes

A research agent searches 3 places **at the same time**. All 3 nodes write to `results` in the same step.
Without a reducer, LangGraph doesn't know which value to keep and raises `InvalidUpdateError`.
With `operator.add`, it simply joins all 3 lists.


In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class ResearchState(TypedDict):
    """Three search nodes run at the same time and all write to ``results``."""

    question: str
    results: Annotated[list[str], operator.add]


def search_web(state: ResearchState) -> dict:
    """Pretend to search the web."""
    return {"results": [f"web: answer for '{state['question']}'"]}


def search_docs(state: ResearchState) -> dict:
    """Pretend to search company documents."""
    return {"results": [f"docs: answer for '{state['question']}'"]}


def search_db(state: ResearchState) -> dict:
    """Pretend to query a database."""
    return {"results": [f"db: answer for '{state['question']}'"]}


def summarize(state: ResearchState) -> dict:
    """Runs after all three searches; sees every result."""
    print(f"summarize sees {len(state['results'])} results")
    return {}


builder = StateGraph(ResearchState)
builder.add_node("search_web", search_web)
builder.add_node("search_docs", search_docs)
builder.add_node("search_db", search_db)
builder.add_node("summarize", summarize)

# Fan-out: START -> 3 searches in parallel, then fan-in -> summarize
for name in ["search_web", "search_docs", "search_db"]:
    builder.add_edge(START, name)
    builder.add_edge(name, "summarize")
builder.add_edge("summarize", END)

builder.compile().invoke({"question": "refund policy", "results": []})
# Without `operator.add`, LangGraph raises InvalidUpdateError:
# three nodes tried to write `results` in the same step and it can't pick one.

### 3b. Custom reducers: add up token cost and merge form fields

A reducer is just a normal function `(old, new) -> combined`. You can write your own.


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


def merge_dicts(old: dict, new: dict) -> dict:
    """Keep old keys, add or update new ones."""
    return {**old, **new}


class OrderState(TypedDict):
    """``total_tokens`` adds up, ``customer`` collects fields from many nodes."""

    total_tokens: Annotated[int, lambda old, new: old + new]
    customer: Annotated[dict, merge_dicts]


def extract_name(state: OrderState) -> dict:
    """Pretend an LLM call found the name and used 120 tokens."""
    return {"customer": {"name": "Asha"}, "total_tokens": 120}


def extract_address(state: OrderState) -> dict:
    """Pretend another LLM call found the city and used 80 tokens."""
    return {"customer": {"city": "Pune"}, "total_tokens": 80}


builder = StateGraph(OrderState)
builder.add_node("extract_name", extract_name)
builder.add_node("extract_address", extract_address)
builder.add_edge(START, "extract_name")
builder.add_edge("extract_name", "extract_address")
builder.add_edge("extract_address", END)

builder.compile().invoke({"total_tokens": 0, "customer": {}})
# -> total_tokens == 200, customer == {"name": "Asha", "city": "Pune"}

### 3c. Trimming chat history to save cost

Long chats send many tokens to the LLM every turn, which costs more money and can hit the context limit.
In production, we often delete old messages with `RemoveMessage`. `add_messages` sees it and removes that id.


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage
from langgraph.graph.message import add_messages

history = [
    HumanMessage("hi", id="1"),
    AIMessage("hello!", id="2"),
    HumanMessage("what is LangGraph?", id="3"),
    AIMessage("A library for building agents as graphs.", id="4"),
]

# Keep only the last 2 messages: send a RemoveMessage for each old one
to_delete = [RemoveMessage(id=m.id) for m in history[:-2]]
history = add_messages(history, to_delete)

print([m.content for m in history])
# -> ['what is LangGraph?', 'A library for building agents as graphs.']

---
✅ **Done with 17 State & reducers.** Next: [18 Tool-calling agent graph →](18_tool_calling_agent_graph.ipynb)  
[🏠 Roadmap](../README.md)